In [34]:
from pathlib import Path
import sys
import pandas as pd
from dotenv import load_dotenv
from plotly.subplots import make_subplots
import plotly.graph_objects as go

ROOT_DIR = Path.cwd().parent

if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.paths import DATA_RAW_PATH, DATA_OUTPUTS_PATH

In [35]:
df_pricing = pd.read_csv(DATA_RAW_PATH / 'get_around_pricing_project.csv', index_col=0)

print(f"DF shape : {df_pricing.shape}")
print()

print(f"The DF for pricing project : ")
display(df_pricing)

DF shape : (4843, 14)

The DF for pricing project : 


,model_key,mileage,engine_power,fuel,paint_color,car_type,private_parking_available,has_gps,has_air_conditioning,automatic_car,has_getaround_connect,has_speed_regulator,winter_tires,rental_price_per_day
0,Citroën,140411,100,diesel,black,convertible,True,True,False,False,True,True,True,106
1,Citroën,13929,317,petrol,grey,convertible,True,True,False,False,False,True,True,264
2,Citroën,183297,120,diesel,white,convertible,False,False,False,False,True,False,True,101
3,Citroën,128035,135,diesel,red,convertible,True,True,False,False,True,True,True,158
4,Citroën,97097,160,diesel,silver,convertible,True,True,False,False,False,True,True,183
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4838,Toyota,39743,110,diesel,black,van,False,True,False,False,False,False,True,121
4839,Toyota,49832,100,diesel,grey,van,False,True,False,False,False,False,True,132
4840,Toyota,19633,110,diesel,grey,van,False,True,False,False,False,False,True,130
4841,Toyota,27920,110,diesel,brown,van,True,True,False,False,False,False,True,151


## 1. Schema validation

In [36]:
display(df_pricing.info())

<class 'pandas.core.frame.DataFrame'>
Index: 4843 entries, 0 to 4842
Data columns (total 14 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   model_key                  4843 non-null   object
 1   mileage                    4843 non-null   int64 
 2   engine_power               4843 non-null   int64 
 3   fuel                       4843 non-null   object
 4   paint_color                4843 non-null   object
 5   car_type                   4843 non-null   object
 6   private_parking_available  4843 non-null   bool  
 7   has_gps                    4843 non-null   bool  
 8   has_air_conditioning       4843 non-null   bool  
 9   automatic_car              4843 non-null   bool  
 10  has_getaround_connect      4843 non-null   bool  
 11  has_speed_regulator        4843 non-null   bool  
 12  winter_tires               4843 non-null   bool  
 13  rental_price_per_day       4843 non-null   int64 
dtypes: bool(7), i

None

In [37]:
int_cols = df_pricing.select_dtypes(include='int64').columns

df_pricing[int_cols] = df_pricing[int_cols].astype("float64")

Schema is structurally sound for ML: 
* int64 changed to float64 : best practice for Mlflow (error shield if any nans residue)
* Features (0-3, 5, 9, 10) and Target (13) are present.
* dtypes are encoding-ready: STR for One-Hot, BOOL for binary, and FLOAT for numeric scaling.

## 2. Duplicates

In [38]:
dups = df_pricing.duplicated().sum()
print(f"Duplicates are : {dups}")

Duplicates are : 0


## 3. Nans

In [39]:
df_pricing.isna().mean().mul(100).round(2).sort_values(ascending=False)

model_key                    0.0
mileage                      0.0
engine_power                 0.0
fuel                         0.0
paint_color                  0.0
car_type                     0.0
private_parking_available    0.0
has_gps                      0.0
has_air_conditioning         0.0
automatic_car                0.0
has_getaround_connect        0.0
has_speed_regulator          0.0
winter_tires                 0.0
rental_price_per_day         0.0
dtype: float64

Dataset is clean. No missing values distinguished.

## 4. Data validity

#### String

In [40]:
str_df = df_pricing.select_dtypes(include=['string'])
str_col = str_df.columns.to_list()

str_unique = {}

for col in str_col:
    str_unique[col] = str_df[col].unique().tolist()

import pprint
pprint.pprint(str_unique)

{}


### Numeric

In [41]:

float_df = df_pricing.select_dtypes(include='float64')
float_col = float_df.columns.to_list()

for col in float_col:
    masked_df = float_df[float_df[col] < 0]
    
    neg_count = len(masked_df) 
    
    print(f"{col}: {neg_count} negative values")

negative_mileage_rows = df_pricing[df_pricing['mileage'] < 0]
display(negative_mileage_rows)

mileage: 1 negative values
engine_power: 0 negative values
rental_price_per_day: 0 negative values


,model_key,mileage,engine_power,fuel,paint_color,car_type,private_parking_available,has_gps,has_air_conditioning,automatic_car,has_getaround_connect,has_speed_regulator,winter_tires,rental_price_per_day
2938,Renault,-64.0,230.0,diesel,black,sedan,True,True,False,True,False,False,True,274.0


In [42]:
print(df_pricing.columns.tolist())

['model_key', 'mileage', 'engine_power', 'fuel', 'paint_color', 'car_type', 'private_parking_available', 'has_gps', 'has_air_conditioning', 'automatic_car', 'has_getaround_connect', 'has_speed_regulator', 'winter_tires', 'rental_price_per_day']


In [43]:
df_pricing = df_pricing[df_pricing['mileage'] >= 0]

negative_check = df_pricing.loc[df_pricing['mileage'] < 0]
print(negative_check)

Empty DataFrame
Columns: [model_key, mileage, engine_power, fuel, paint_color, car_type, private_parking_available, has_gps, has_air_conditioning, automatic_car, has_getaround_connect, has_speed_regulator, winter_tires, rental_price_per_day]
Index: []


## 5. Outliers

In [44]:
fig = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=float_col
)

fig.add_trace(
    go.Box(y=df_pricing[float_col[0]], name=float_col[0]),
    row=1,
    col=1
)

fig.add_trace(
    go.Box(y=df_pricing[float_col[1]], name=float_col[1]),
    row=1,
    col=2
)

fig.add_trace(
    go.Box(y=df_pricing[float_col[2]], name=float_col[2]),
    row=1,
    col=3
)

fig.show()

In [45]:
# Not being too strict so ML is not biased, can cope better with real data

condition_mileage = df_pricing['mileage'] < 500_000                
condition_power   = (df_pricing['engine_power'] > 10) & (df_pricing['engine_power'] < 400)
condition_price   = df_pricing['rental_price_per_day'] > 10          

df_clean = df_pricing[condition_mileage & condition_power & condition_price]

print(f"Lignes avant : {len(df_pricing)}")
print(f"Lignes après : {len(df_clean)}")
print(f"Lignes supprimées : {len(df_pricing) - len(df_clean)}")

Lignes avant : 4842
Lignes après : 4831
Lignes supprimées : 11


In [46]:
df_clean.to_csv(DATA_OUTPUTS_PATH / "pricing_data.csv")